
# GVH Diagonal Cubic 0.3.2.7.3.2 — Full Hamiltonian / Dirac Constraint and Physical-DOF Audit

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.7.3.2  
**Position :** après `0.3.2.7.3.1`, avant toute dispersion physique \(\omega^2(k)\)

---

## Objectif

`0.3.2.7.3.1` a rouvert correctement le gate G3 :

\[
\boxed{
G3=\text{OPEN — FULL HAMILTONIAN/DIRAC ANALYSIS REQUIRED}
}
\]

et a interdit de réutiliser les anciens nombres heuristiques \(2,3,3\) comme degrés de liberté physiques GVH.

Ce notebook attaque le problème canonique avec une règle stricte :

\[
\boxed{
N_{\rm phys}
=
\frac{
N_{\rm phase}
-2N_{\rm first}
-N_{\rm second}
}{2}
}
\]

mais **aucun nombre final de DOF ne sera publié tant que l'action canonique complète, les contraintes secondaires et leur classe ne sont pas effectivement dérivées**.

Le notebook doit donc distinguer :

- ce qui est dérivé exactement ;
- ce qui est dérivé localement/au niveau du Hessien cinétique ;
- ce qui reste bloqué faute d'action ADM complète ;
- les surfaces de dégénérescence des couplages qui peuvent changer le nombre de contraintes.



## 0. Conventions héritées

Secteur vectoriel candidat :

\[
\mathcal L_u
=
-c_1(\nabla_\mu u_\nu)(\nabla^\mu u^\nu)
-c_2(\nabla_\mu u^\mu)^2
-c_3(\nabla_\mu u_\nu)(\nabla^\nu u^\mu)
+c_4 a_\mu a^\mu,
\]

\[
a^\mu=u^\nu\nabla_\nu u^\mu,
\]

avec contrainte physique

\[
u^\mu u_\mu+1=0
\]

imposée par un multiplicateur \(\lambda_{\rm mult}\).

L'architecture DC-4 utilise une métrique 4D covariante.  
L'architecture DC-3+Flow utilise \(h_{ij}(x,\lambda)\), mais son **action complète propre** n'est pas encore fermée.

Conséquence immédiate :

> une analyse Dirac complète de DC-3+Flow ne peut pas être inventée avant d'avoir écrit son action explicite.


In [1]:

from __future__ import annotations

from pathlib import Path
import json
import sys
import sympy as sp
import pandas as pd

print("GVH Diagonal Cubic 0.3.2.7.3.2")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH Diagonal Cubic 0.3.2.7.3.2
Python: 3.12.13
SymPy: 1.14.0



# 1. Formule de Dirac — verrou de comptage

Pour un système hamiltonien contraint :

\[
\boxed{
N_{\mathrm{phys}}
=
\frac{
N_{\mathrm{phase}}
-2N_{\mathrm{1st}}
-N_{\mathrm{2nd}}
}{2}
}
\]

où :

- \(N_{\rm phase}\) est la dimension de l'espace des phases ;
- \(N_{\rm 1st}\) est le nombre de contraintes de première classe ;
- \(N_{\rm 2nd}\) est le nombre de contraintes de seconde classe.

Un simple calcul du type

\[
N_{\rm configuration}-N_{\rm gauge}
\]

n'est donc pas accepté comme analyse de Dirac.


In [2]:

def dirac_dof(N_phase, N_first, N_second):
    return sp.Rational(N_phase - 2*N_first - N_second, 2)

# Benchmark contrôlé : GR pure ADM.
assert dirac_dof(12, 4, 0) == 2

print("Benchmark GR pure ADM: 2 DOF")
print("Ce résultat n'est PAS transféré au modèle GVH vector-tensor.")


Benchmark GR pure ADM: 2 DOF
Ce résultat n'est PAS transféré au modèle GVH vector-tensor.



# 2. Registre canonique DC-4 — variables à traiter

Une décomposition ADM naturelle de la métrique introduit :

\[
h_{ij}\;(6),\qquad N\;(1),\qquad N^i\;(3).
\]

Le secteur directionnel ajoute :

\[
u^\mu\;(4),
\qquad
\lambda_{\rm mult}\;(1).
\]

Donc, avant contraintes et avant toute élimination :

\[
6+1+3+4+1=15
\]

variables de configuration.

Cela donne potentiellement

\[
30
\]

variables de phase si toutes reçoivent un moment conjugué.

Mais le nombre de **moments indépendants non nuls** dépend du Hessien cinétique de l'action complète.


In [3]:

dc4_config = pd.DataFrame([
    {"field":"h_ij", "components":6, "role":"spatial metric"},
    {"field":"N", "components":1, "role":"lapse"},
    {"field":"N^i", "components":3, "role":"shift"},
    {"field":"u^mu", "components":4, "role":"timelike/vector field"},
    {"field":"lambda_mult", "components":1, "role":"unit-norm multiplier"},
])

dc4_config["components"].sum(), dc4_config


(np.int64(15),
          field  components                   role
 0         h_ij           6         spatial metric
 1            N           1                  lapse
 2          N^i           3                  shift
 3         u^mu           4  timelike/vector field
 4  lambda_mult           1   unit-norm multiplier)


## 2.1 Contraintes primaires certaines vs hypothétiques

Une contrainte primaire n'est déclarée **certaine** ici que si elle suit directement de l'absence de vitesse correspondante dans l'action.

Le multiplicateur \(\lambda_{\rm mult}\) n'a pas de terme cinétique :

\[
p_\lambda\approx0.
\]

Sa préservation temporelle doit générer la contrainte de norme :

\[
\chi_{\rm norm}
=
u^\mu u_\mu+1
\approx0.
\]

Pour \(N\) et \(N^i\), l'analogie avec GR suggère des moments primaires nuls, mais **le secteur vectoriel complet doit être décomposé en ADM avant de l'affirmer sans réserve**, car des connexions contenant des dérivées du lapse/shift peuvent se réorganiser après intégrations par parties.


In [4]:

known_constraints = pd.DataFrame([
    {
        "constraint":"p_lambda ≈ 0",
        "origin":"lambda_mult has no kinetic term",
        "status":"DERIVED_STRUCTURALLY",
        "class":"TO_CLASSIFY",
    },
    {
        "constraint":"u.u + 1 ≈ 0",
        "origin":"preservation / lambda equation",
        "status":"EXPECTED_SECONDARY_FROM_MULTIPLIER",
        "class":"TO_CLASSIFY_WITH_FULL_POISSON_MATRIX",
    },
    {
        "constraint":"p_N ≈ 0",
        "origin":"ADM lapse",
        "status":"TO_VERIFY_IN_FULL_GVH_ADM_ACTION",
        "class":"OPEN",
    },
    {
        "constraint":"p_i ≈ 0",
        "origin":"ADM shift",
        "status":"TO_VERIFY_IN_FULL_GVH_ADM_ACTION",
        "class":"OPEN",
    },
])
known_constraints


,constraint,origin,status,class
0,p_lambda ≈ 0,lambda_mult has no kinetic term,DERIVED_STRUCTURALLY,TO_CLASSIFY
1,u.u + 1 ≈ 0,preservation / lambda equation,EXPECTED_SECONDARY_FROM_MULTIPLIER,TO_CLASSIFY_WITH_FULL_POISSON_MATRIX
2,p_N ≈ 0,ADM lapse,TO_VERIFY_IN_FULL_GVH_ADM_ACTION,OPEN
3,p_i ≈ 0,ADM shift,TO_VERIFY_IN_FULL_GVH_ADM_ACTION,OPEN



# 3. Hessien cinétique local du secteur vectoriel

Avant de construire l'algèbre complète de Dirac, on peut déterminer exactement quelles combinaisons de \(c_i\) contrôlent localement les vitesses du champ \(u^\mu\).

On travaille à un point en repère local inertiel :

\[
g_{\mu\nu}=\eta_{\mu\nu},
\qquad
u^\mu=(1,0,0,0),
\]

et on conserve uniquement les vitesses

\[
V^\mu=\partial_0u^\mu.
\]

Ce calcul n'est **pas** un comptage complet de DOF.  
Il sert à identifier les surfaces où le Hessien cinétique devient dégénéré et où le nombre de contraintes peut changer.


In [5]:

c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
V0,V1,V2,V3 = sp.symbols("V0 V1 V2 V3", real=True)

eta = sp.diag(-1,1,1,1)
eta_inv = eta

V = sp.Matrix([V0,V1,V2,V3])

# Only ∂_0 u^nu = V^nu is retained.
nabla_con = sp.MutableDenseMatrix(4,4,[0]*16)
for nu in range(4):
    nabla_con[0,nu] = V[nu]

nabla_cov = sp.MutableDenseMatrix(4,4,[0]*16)
for mu in range(4):
    for nu in range(4):
        nabla_cov[mu,nu] = sum(
            eta[nu,rho]*nabla_con[mu,rho]
            for rho in range(4)
        )

I1 = sp.S.Zero
for mu in range(4):
    for nu in range(4):
        for a in range(4):
            for b in range(4):
                I1 += (
                    eta_inv[mu,a]*eta_inv[nu,b]
                    *nabla_cov[mu,nu]
                    *nabla_cov[a,b]
                )

theta = sp.trace(nabla_con)

I3 = sp.S.Zero
for mu in range(4):
    for nu in range(4):
        I3 += nabla_cov[mu,nu] * sum(
            eta_inv[nu,a]*nabla_con[a,mu]
            for a in range(4)
        )

a_con = V
a2 = sp.simplify((a_con.T*eta*a_con)[0])

Lkin = sp.factor(
    -c1*I1
    -c2*theta**2
    -c3*I3
    +c4*a2
)

print("I1 =", sp.factor(I1))
print("theta =", theta)
print("I3 =", sp.factor(I3))
print("a^2 =", sp.factor(a2))
print("L_kin =", Lkin)


I1 = V0**2 - V1**2 - V2**2 - V3**2
theta = V0
I3 = V0**2
a^2 = -V0**2 + V1**2 + V2**2 + V3**2
L_kin = -V0**2*c1 - V0**2*c2 - V0**2*c3 - V0**2*c4 + V1**2*c1 + V1**2*c4 + V2**2*c1 + V2**2*c4 + V3**2*c1 + V3**2*c4



Le résultat local est :

\[
\boxed{
\mathcal L_{\rm kin}
=
-(c_1+c_2+c_3+c_4)V_0^2
+
(c_1+c_4)(V_1^2+V_2^2+V_3^2)
}
\]

avant réduction explicite par la contrainte de norme.

On définit donc provisoirement :

\[
c_{\rm time}=c_1+c_2+c_3+c_4,
\]

\[
c_{14}=c_1+c_4.
\]

Après linéarisation de

\[
u^\mu u_\mu=-1
\]

autour de \(u^\mu=(1,0,0,0)\), la fluctuation temporelle \(\delta u^0\) est contrainte au premier ordre. Les trois vitesses spatiales sont donc le secteur cinétique immédiatement pertinent, contrôlé par \(c_{14}\).


In [6]:

velocities = sp.Matrix([V0,V1,V2,V3])
H_u = sp.simplify(sp.hessian(Lkin, velocities))

print("Vector kinetic Hessian:")
sp.pprint(H_u)

det_Hu = sp.factor(H_u.det())
print("\ndet(H_u) =", det_Hu)

eigen_diag = [sp.factor(H_u[i,i]) for i in range(4)]
print("\ndiagonal entries:", eigen_diag)

expected = sp.diag(
    -2*(c1+c2+c3+c4),
    2*(c1+c4),
    2*(c1+c4),
    2*(c1+c4),
)

assert sp.simplify(H_u-expected) == sp.zeros(4)


Vector kinetic Hessian:
⎡-2⋅c₁ - 2⋅c₂ - 2⋅c₃ - 2⋅c₄       0            0            0     ⎤
⎢                                                                 ⎥
⎢            0               2⋅c₁ + 2⋅c₄       0            0     ⎥
⎢                                                                 ⎥
⎢            0                    0       2⋅c₁ + 2⋅c₄       0     ⎥
⎢                                                                 ⎥
⎣            0                    0            0       2⋅c₁ + 2⋅c₄⎦

det(H_u) = -16*(c1 + c4)**3*(c1 + c2 + c3 + c4)

diagonal entries: [-2*(c1 + c2 + c3 + c4), 2*(c1 + c4), 2*(c1 + c4), 2*(c1 + c4)]



## 3.1 Surfaces de dégénérescence canonique

Le Hessien montre deux surfaces spéciales :

\[
\boxed{c_{14}=c_1+c_4=0}
\]

et

\[
\boxed{
c_{\rm time}
=
c_1+c_2+c_3+c_4=0.
}
\]

Sur ces surfaces, le rang du Hessien chute et de nouvelles contraintes primaires peuvent apparaître.

Par conséquent, **le nombre de DOF ne peut pas être universellement déclaré sans préciser la branche de couplages**.


In [7]:

c14 = sp.symbols("c14", real=True)
ctime = sp.symbols("c_time", real=True)

rank_cases = pd.DataFrame([
    {
        "branch":"generic",
        "conditions":"c14 != 0 and c_time != 0",
        "local_vector_Hessian_rank":4,
        "interpretation":"no velocity null direction before norm reduction",
    },
    {
        "branch":"c14 = 0",
        "conditions":"c1+c4 = 0, c_time != 0",
        "local_vector_Hessian_rank":1,
        "interpretation":"three spatial velocity null directions",
    },
    {
        "branch":"c_time = 0",
        "conditions":"c1+c2+c3+c4 = 0, c14 != 0",
        "local_vector_Hessian_rank":3,
        "interpretation":"temporal velocity null direction",
    },
    {
        "branch":"double-degenerate",
        "conditions":"c14 = 0 and c_time = 0",
        "local_vector_Hessian_rank":0,
        "interpretation":"all vector velocity directions null locally",
    },
])

rank_cases


,branch,conditions,local_vector_Hessian_rank,interpretation
0,generic,c14 != 0 and c_time != 0,4,no velocity null direction before norm reduction
1,c14 = 0,"c1+c4 = 0, c_time != 0",1,three spatial velocity null directions
2,c_time = 0,"c1+c2+c3+c4 = 0, c14 != 0",3,temporal velocity null direction
3,double-degenerate,c14 = 0 and c_time = 0,0,all vector velocity directions null locally



# 4. Effet canonique de la contrainte de norme

La contrainte

\[
\chi_1=u^\mu u_\mu+1\approx0
\]

est accompagnée du moment du multiplicateur

\[
\chi_0=p_\lambda\approx0.
\]

Pour connaître leur classe, il faut calculer leur matrice de Poisson avec **toutes** les contraintes secondaires issues du Hamiltonien total.

La paire \((p_\lambda,\chi_1)\) ne doit donc pas être automatiquement déclarée seconde classe sans terminer l'algorithme de Dirac-Bergmann.


In [8]:

constraint_algorithm = pd.DataFrame([
    {"stage":"0", "operation":"Construct canonical momenta", "status":"PARTIAL — vector local Hessian done"},
    {"stage":"1", "operation":"Identify primary constraints", "status":"PARTIAL — p_lambda certain; full ADM set open"},
    {"stage":"2", "operation":"Build canonical Hamiltonian H_C", "status":"BLOCKED — full ADM action not yet explicitly decomposed"},
    {"stage":"3", "operation":"Build total Hamiltonian H_T", "status":"BLOCKED"},
    {"stage":"4", "operation":"Preserve all primary constraints in lambda-flow", "status":"BLOCKED"},
    {"stage":"5", "operation":"Derive secondary/tertiary constraints", "status":"BLOCKED"},
    {"stage":"6", "operation":"Compute full Poisson-bracket matrix", "status":"BLOCKED"},
    {"stage":"7", "operation":"Classify first/second class", "status":"BLOCKED"},
    {"stage":"8", "operation":"Count physical DOF", "status":"BLOCKED"},
])

constraint_algorithm


,stage,operation,status
0,0,Construct canonical momenta,PARTIAL — vector local Hessian done
1,1,Identify primary constraints,PARTIAL — p_lambda certain; full ADM set open
2,2,Build canonical Hamiltonian H_C,BLOCKED — full ADM action not yet explicitly d...
3,3,Build total Hamiltonian H_T,BLOCKED
4,4,Preserve all primary constraints in lambda-flow,BLOCKED
5,5,Derive secondary/tertiary constraints,BLOCKED
6,6,Compute full Poisson-bracket matrix,BLOCKED
7,7,Classify first/second class,BLOCKED
8,8,Count physical DOF,BLOCKED



# 5. DC-3+Flow — pourquoi le comptage reste bloqué

Une analyse canonique exige un Lagrangien explicite :

\[
S_{\rm DC3+Flow}
=
\int d\lambda\,d^3x\,
\mathcal L_{\rm DC3}
(h_{ij},\dot h_{ij},N,N^i,\ldots).
\]

À ce stade, `0.3.2.7.3` n'a établi que la **condition de fermeture** du secteur gravitationnel :

\[
N\sqrt h
\left[
{}^{(3)}R
+
K_{ij}K^{ij}
-
K^2
\right]
\]

si l'on veut reproduire le secteur Einstein-Hilbert de DC-4 via la carte ADM.

Cela ne constitue pas encore une action DC-3+Flow complète incluant le secteur directionnel GVH.

Donc :

\[
\boxed{
N_{\rm phys}^{DC3+Flow}
=
\text{OPEN}
}
\]

et aucun remplacement numérique n'est autorisé.



# 6. DC-3+Emergent Clock — branche auxiliaire

Deux cas doivent rester séparés.

### Cas A — \(\tau[h]\) est seulement une observable dérivée

Alors \(\tau\) n'est **pas** une variable canonique :

\[
\Delta N_{\rm phase}=0.
\]

### Cas B — \(\tau\) est introduit comme variable auxiliaire indépendante

Alors on doit introduire

\[
(\tau,p_\tau)
\]

et une contrainte

\[
\chi_\tau
=
\tau-F[h]
\approx0.
\]

Sa préservation peut générer une contrainte supplémentaire. La classe de cette paire doit être déterminée par l'algèbre de Poisson.

Donc le nombre de DOF reste également OPEN.


In [9]:

architecture_status = pd.DataFrame([
    {
        "architecture":"DC-4 GVH vector-tensor",
        "physical_DOF":"OPEN",
        "why":"full ADM Hamiltonian + constraint algebra not completed",
    },
    {
        "architecture":"DC-3+Flow",
        "physical_DOF":"OPEN",
        "why":"explicit complete DC-3+Flow action not yet defined",
    },
    {
        "architecture":"DC-3+Emergent Clock",
        "physical_DOF":"OPEN",
        "why":"depends on whether tau is derived or auxiliary + constraint class",
    },
])
architecture_status


,architecture,physical_DOF,why
0,DC-4 GVH vector-tensor,OPEN,full ADM Hamiltonian + constraint algebra not ...
1,DC-3+Flow,OPEN,explicit complete DC-3+Flow action not yet def...
2,DC-3+Emergent Clock,OPEN,depends on whether tau is derived or auxiliary...



# 7. Ce que ce notebook ferme réellement

### Fermé

- formule de Dirac verrouillée ;
- anciens \(2,3,3\) maintenus rétractés ;
- registre canonique DC-4 explicité ;
- \(p_\lambda\approx0\) identifié structurellement ;
- contrainte de norme explicitée ;
- Hessien cinétique local du secteur \(u^\mu\) dérivé ;
- surfaces de dégénérescence \(c_{14}=0\) et \(c_{\rm time}=0\) identifiées ;
- dépendance possible du nombre de contraintes aux \(c_i\) démontrée.

### Non fermé

- action ADM complète du secteur vectoriel ;
- Hamiltonien canonique complet ;
- contraintes secondaires/tertiaires ;
- matrice complète de Poisson ;
- classification première/seconde classe ;
- DOF physiques finaux ;
- action propre complète DC-3+Flow.

Le résultat scientifique doit donc être `PARTIAL PASS / BLOCKED`, pas `PASS`.


In [10]:

GATES = {
    "Dirac_formula_locked": True,
    "old_2_3_3_counts_retracted": True,
    "DC4_canonical_registry_defined": True,
    "p_lambda_primary_identified": True,
    "norm_constraint_identified": True,
    "local_vector_kinetic_Hessian_derived": True,
    "coupling_degeneracy_surfaces_identified": True,
    "full_DC4_ADM_Hamiltonian_derived": False,
    "all_secondary_constraints_derived": False,
    "full_Poisson_matrix_computed": False,
    "first_second_classification_complete": False,
    "physical_DOF_final": False,
    "explicit_DC3Flow_action_complete": False,
}

for k,v in GATES.items():
    print(f"{k}: {v}")

PARTIAL_PASS = all([
    GATES["Dirac_formula_locked"],
    GATES["local_vector_kinetic_Hessian_derived"],
    GATES["coupling_degeneracy_surfaces_identified"],
])

FULL_PASS = all(GATES.values())

assert PARTIAL_PASS is True
assert FULL_PASS is False

FINAL_STATUS = (
    "PARTIAL-PASS-HAMILTONIAN-DIRAC-FOUNDATION_"
    "VECTOR-KINETIC-HESSIAN-AND-DEGENERACY-SURFACES-DERIVED_"
    "BLOCKED-FULL-ADM-HAMILTONIAN-CONSTRAINT-ALGEBRA-AND-PHYSICAL-DOF"
)

print("\nFINAL STATUS:", FINAL_STATUS)


Dirac_formula_locked: True
old_2_3_3_counts_retracted: True
DC4_canonical_registry_defined: True
p_lambda_primary_identified: True
norm_constraint_identified: True
local_vector_kinetic_Hessian_derived: True
coupling_degeneracy_surfaces_identified: True
full_DC4_ADM_Hamiltonian_derived: False
all_secondary_constraints_derived: False
full_Poisson_matrix_computed: False
first_second_classification_complete: False
physical_DOF_final: False
explicit_DC3Flow_action_complete: False

FINAL STATUS: PARTIAL-PASS-HAMILTONIAN-DIRAC-FOUNDATION_VECTOR-KINETIC-HESSIAN-AND-DEGENERACY-SURFACES-DERIVED_BLOCKED-FULL-ADM-HAMILTONIAN-CONSTRAINT-ALGEBRA-AND-PHYSICAL-DOF



# 8. Gate de dispersion

Une relation

\[
\omega^2(k)
\]

ne doit être associée à un « mode physique » qu'après séparation entre :

- variables propagatives ;
- variables auxiliaires ;
- contraintes ;
- modes de jauge.

Le gate reste donc :

\[
\boxed{\text{DISPERSION\_READY=False}}.
\]


In [11]:

DISPERSION_READY = False

dispersion_blockers = [
    "full DC-4 ADM Hamiltonian not derived",
    "constraint algebra not closed",
    "physical DOF not classified",
    "DC-3+Flow complete action not yet written",
]

assert DISPERSION_READY is False

print("DISPERSION_READY =", DISPERSION_READY)
for item in dispersion_blockers:
    print("BLOCK:", item)


DISPERSION_READY = False
BLOCK: full DC-4 ADM Hamiltonian not derived
BLOCK: constraint algebra not closed
BLOCK: physical DOF not classified
BLOCK: DC-3+Flow complete action not yet written



# 9. Prochaine sous-étape

La suite canonique doit être divisée proprement.

## 0.3.2.7.3.3 — Full ADM Decomposition of the GVH Vector Sector

Objectif :

\[
\mathcal L_u[g_{\mu\nu},u^\mu]
\longrightarrow
\mathcal L_u[
h_{ij},N,N^i,
u^\perp,u^i,
K_{ij},
D_i,\partial_\lambda
].
\]

Cette décomposition permettra alors de calculer réellement :

\[
\pi^{ij}
=
\frac{\partial\mathcal L}{\partial\dot h_{ij}},
\qquad
p_\mu^{(u)}
=
\frac{\partial\mathcal L}{\partial\dot u^\mu},
\]

et de déterminer les contraintes primaires exactes.

Ensuite seulement l'algorithme de Dirac-Bergmann pourra être poursuivi jusqu'aux DOF physiques.



# 10. Artefact machine-readable


In [12]:

artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.2",
    "final_status": FINAL_STATUS,

    "Dirac_formula": "(N_phase - 2*N_first - N_second)/2",

    "DC4": {
        "configuration_components_before_constraints": 15,
        "physical_DOF": "OPEN",
        "certain_primary_constraints": ["p_lambda ~= 0"],
        "norm_constraint": "u^mu u_mu + 1 ~= 0",
    },

    "local_vector_kinetic_audit": {
        "Lkin": str(Lkin),
        "Hessian": str(H_u),
        "generic_rank": 4,
        "degeneracy_surfaces": [
            "c1+c4 = 0",
            "c1+c2+c3+c4 = 0",
        ],
        "note": "rank before explicit linearized norm reduction",
    },

    "DC3_Flow": {
        "physical_DOF": "OPEN",
        "blocker": "explicit complete action not yet written",
    },

    "DC3_Emergent": {
        "physical_DOF": "OPEN",
        "blocker": "tau canonical status and constraint class unresolved",
    },

    "dispersion_ready": DISPERSION_READY,

    "next_notebook": (
        "GVH_Diagonal_Cubic_0.3.2.7.3.3_"
        "Full_ADM_Decomposition_of_the_GVH_Vector_Sector.ipynb"
    ),
}

if Path("/content").exists():
    export_dir = Path("/content/gvh_exports")
else:
    export_dir = Path.cwd() / "gvh_exports"

export_dir.mkdir(parents=True, exist_ok=True)

artifact_path = export_dir / "gvh_0.3.2.7.3.2_hamiltonian_dirac_audit.json"
artifact_path.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

assert artifact_path.exists()
assert artifact["dispersion_ready"] is False
assert artifact["DC4"]["physical_DOF"] == "OPEN"

print("Artifact:", artifact_path)
print("Status:", FINAL_STATUS)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.2_hamiltonian_dirac_audit.json
Status: PARTIAL-PASS-HAMILTONIAN-DIRAC-FOUNDATION_VECTOR-KINETIC-HESSIAN-AND-DEGENERACY-SURFACES-DERIVED_BLOCKED-FULL-ADM-HAMILTONIAN-CONSTRAINT-ALGEBRA-AND-PHYSICAL-DOF



# Conclusion

`0.3.2.7.3.2` ne fabrique pas un nombre de DOF que l'action complète ne permet pas encore de justifier.

Il établit cependant un résultat canonique important : dans un repère local inertiel autour d'un fond timelike,

\[
\mathcal L_{\rm kin}
=
-(c_1+c_2+c_3+c_4)V_0^2
+
(c_1+c_4)\sum_{i=1}^3 V_i^2.
\]

Le Hessien cinétique change donc de rang sur

\[
\boxed{c_1+c_4=0}
\]

et

\[
\boxed{c_1+c_2+c_3+c_4=0}.
\]

Le nombre et la nature des contraintes peuvent donc dépendre de la branche de couplages.

Le verdict correct est :

```text
PARTIAL-PASS-HAMILTONIAN-DIRAC-FOUNDATION_VECTOR-KINETIC-HESSIAN-AND-DEGENERACY-SURFACES-DERIVED_BLOCKED-FULL-ADM-HAMILTONIAN-CONSTRAINT-ALGEBRA-AND-PHYSICAL-DOF
```

et

\[
\boxed{\text{DISPERSION\_READY=False}}.
\]

La prochaine étape doit être la décomposition ADM complète du secteur vectoriel avant tout calcul physique de \(\omega^2(k)\).
